<a href="https://colab.research.google.com/github/lucaser9898/sovereign-funds-rl-portfolio/blob/main/A2C_tesi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# A2C Portfolio

# =========================
# 0) Install (facoltativo per Colab)
# =========================
try:
    import google.colab
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    !pip -q install numpy pandas matplotlib torch tqdm

# =========================
# 1) Parameters
# =========================
from dataclasses import dataclass
from typing import Tuple, List, NamedTuple, Optional, Dict
import itertools, random
import numpy as np
import pandas as pd

# Dati & split
WINDOW = 12
PERIODS_PER_YEAR = 12
DATE_COL = None

# Costi / Cash
TRANS_COST_BPS = 10
HYSTERESIS_BPS = 10
INCLUDE_CASH = True
RF_PER_PERIOD = 0.0

# RL
SEED = 42
GAMMA = 0.99
LR = 3e-4
VALUE_COEF = 0.5
ENTROPY_COEF = 1e-3
MAX_GRAD_NORM = 1.0
N_STEPS = 32
TOTAL_UPDATES = 10000

# GAE
USE_GAE = True
GAE_LAMBDA = 0.95

# Tilt bounds (relativi ai pesi base)
USE_TILT_BOUNDS = True
SIZE_THRESH = 0.10
TILT_MIN_SMALL = 0.50
TILT_MAX_SMALL = 2.00
TILT_MIN_LARGE = 0.50
TILT_MAX_LARGE = 1.50
TILT_MIN_CASH  = 0.50
TILT_MAX_CASH  = 1.50

# Rischio (EWMA)
RISK_AV   = 0.5
COV_WIN   = 12
RISK_MODE = "full"

# Salvataggio
MODEL_PATH = "a2c_portafoglio_model.pt"

# =========================
# 2) Upload
# =========================
if IN_COLAB:
    from google.colab import files
    print("Seleziona due CSV (PREZZI & PESI). I nomi possono essere qualsiasi.")
    uploaded = files.upload()
    if len(uploaded) < 2:
        raise FileNotFoundError("Carica almeno due CSV.")
else:
    uploaded = {"indici_borsa_filtrati.csv": None, "pesi_ritracciati.csv": None}

dfs = {}
for fname in uploaded.keys():
    dfs[fname] = pd.read_csv(fname)

# =========================
# 3) Preprocessing
# =========================

def to_numeric_df(df: pd.DataFrame) -> pd.DataFrame:
    """Converte tutto a numerico, rimuove colonne vuote, sostituisce inf con NaN e fa ffill/bfill.
    NIENTE fillna(0.0) qui: verrà fatto solo sui pesi."""
    df_num = df.copy()
    for c in df_num.columns:
        df_num[c] = pd.to_numeric(df_num[c], errors="coerce")
    df_num = df_num.dropna(axis=1, how="all")
    df_num = df_num.replace([np.inf, -np.inf], np.nan)
    return df_num.ffill().bfill()

def pick_prices_weights(dfs_dict: dict) -> Tuple[str, pd.DataFrame, str, pd.DataFrame]:
    """Heuristica robusta: massima sovrapposizione colonne; il 'prezzi' è quello con più righe e più VARIANZA."""
    best_pair, best_overlap, best_score = None, -1, -1.0
    names = list(dfs_dict.keys())
    for a, b in itertools.combinations(names, 2):
        cols_a, cols_b = set(dfs_dict[a].columns), set(dfs_dict[b].columns)
        common = list(cols_a & cols_b)
        ov = len(common)
        if ov == 0:
            continue

        def score_prices(df):
            num = to_numeric_df(df[common])
            var = float(num.var(numeric_only=True).sum(skipna=True))
            rows = df.shape[0]
            return var * np.log1p(rows)
        sa, sb = score_prices(dfs_dict[a]), score_prices(dfs_dict[b])
        if ov > best_overlap or (ov == best_overlap and max(sa, sb) > best_score):
            best_overlap = ov
            best_score = max(sa, sb)
            if sa >= sb:
                best_pair = (a, b)  #
            else:
                best_pair = (b, a)
    if not best_pair:
        # fallback
        names = list(dfs_dict.keys())
        best_pair = (names[0], names[1])
    a, b = best_pair
    return a, dfs_dict[a], b, dfs_dict[b]

def normalize_dates(df: pd.DataFrame, date_col: Optional[str]) -> pd.DataFrame:
    df = df.copy()
    cols_lower = {c.lower(): c for c in df.columns}
    guess = None
    for k in ["date", "data", "time", "timestamp"]:
        if k in cols_lower:
            guess = cols_lower[k]
            break
    dcol = date_col or guess
    if dcol and dcol in df.columns:
        df[dcol] = pd.to_datetime(df[dcol], errors="coerce")
        df = df.dropna(subset=[dcol])
        df = df.set_index(dcol)
    df = df.sort_index()
    return df

prices_name, df_prices_raw, weights_name, df_weights_ref_raw = pick_prices_weights(dfs)
print(f"Dati ✅  PREZZI: {prices_name} {df_prices_raw.shape}  |  PESI: {weights_name} {df_weights_ref_raw.shape}")

df_prices_raw = normalize_dates(df_prices_raw, DATE_COL)
df_weights_ref_raw = normalize_dates(df_weights_ref_raw, DATE_COL)

_dfp = to_numeric_df(df_prices_raw)
_dfw = to_numeric_df(df_weights_ref_raw)


common_cols = sorted(set(_dfp.columns) & set(_dfw.columns))
if len(common_cols) < 2:
    common_cols = list(_dfp.columns)

prices = _dfp[common_cols].copy().sort_index()
weights_ref = _dfw[common_cols].copy().fillna(0.0)


rets = prices.pct_change().replace([np.inf, -np.inf], np.nan).fillna(0.0)

# Pesi base dalla PRIMA riga dei pesi (cash corretto)
raw_base = weights_ref.iloc[0].astype(float).clip(lower=0).fillna(0.0)
s = float(raw_base.sum())
if INCLUDE_CASH:
    if s <= 1.0 and s > 0:
        w_base_assets = raw_base.values
        base_cash = 1.0 - s
    elif s == 0:
        w_base_assets = np.ones(len(common_cols)) / len(common_cols)
        base_cash = 0.0
    else:
        w_base_assets = (raw_base / s).values
        base_cash = 0.0
else:
    w_base_assets = (raw_base / s).values if s > 0 else np.ones(len(common_cols)) / len(common_cols)
    base_cash = 0.0

# Split 70/15/15 (time-ordered)
n = len(rets)
i_train = int(n * 0.70)
i_val = int(n * 0.85)
rets_train, rets_val, rets_test = rets.iloc[:i_train], rets.iloc[i_train:i_val], rets.iloc[i_val:]

# Standardizzazione SENZA leakage: fit su train
mu_tr, sigma_tr = rets_train.mean(), rets_train.std()
sigma_tr = sigma_tr.replace(0, 1.0)

def zscore(df: pd.DataFrame) -> pd.DataFrame:
    return ((df - mu_tr) / sigma_tr).replace([np.inf, -np.inf], np.nan).fillna(0.0)

retsn_train = zscore(rets_train)
retsn_val = zscore(rets_val)
retsn_test = zscore(rets_test)
rets_norm_all = pd.concat([retsn_train, retsn_val, retsn_test])

print(f"Train={len(rets_train)}  Val={len(rets_val)}  Test={len(rets_test)}  | Assets={len(common_cols)}")

# =========================
# 4) Bounds & Proiezione
# =========================
def build_tilt_bounds(
    w_base_assets: np.ndarray,
    base_cash: float,
    include_cash: bool,
    size_thresh: float,
    tmin_small: float,
    tmax_small: float,
    tmin_large: float,
    tmax_large: float,
    tmin_cash: float,
    tmax_cash: float,
):
    n = len(w_base_assets)
    lb = np.zeros(n + (1 if include_cash else 0), dtype=float)
    ub = np.zeros_like(lb)
    for i, wb in enumerate(w_base_assets):
        if wb >= size_thresh:
            lb[i] = wb * tmin_large
            ub[i] = wb * tmax_large
        else:
            lb[i] = wb * tmin_small
            ub[i] = wb * tmax_small
    eps = 1e-6
    if include_cash:
        lb[-1] = max(eps, base_cash * tmin_cash)
        ub[-1] = max(lb[-1] + eps, base_cash * tmax_cash)
    # Feasibility fix
    s_lb, s_ub = lb.sum(), ub.sum()
    if s_lb > 1.0:
        lb = lb / s_lb
    if s_ub < 1.0:
        scale = (1.0 / max(s_ub, 1e-12))
        ub = np.maximum(ub * scale, lb + 1e-12)
    return lb, ub

def project_bounded_simplex(v, lb, ub, s: float = 1.0, max_iter: int = 200):
    """Proiezione iterativa su simplex con bounds; semplice e robusta per questo contesto."""
    w = np.clip(v, lb, ub).astype(float)
    for _ in range(max_iter):
        diff = s - w.sum()
        if abs(diff) < 1e-12:
            break
        if diff > 0:
            free = (w < ub - 1e-12)
            if not np.any(free): break
            w[free] = np.minimum(ub[free], w[free] + diff / free.sum())
        else:
            free = (w > lb + 1e-12)
            if not np.any(free): break
            w[free] = np.maximum(lb[free], w[free] + diff / free.sum())
    if abs(w.sum() - s) > 1e-10:
        free = (w > lb + 1e-12) & (w < ub - 1e-12)
        if np.any(free):
            w[free] += (s - w.sum()) / free.sum()
            w = np.clip(w, lb, ub)
    return w

# =========================
# 5) Ambiente — stateful, isteresi, costi, RF cash, EWMA risk
# =========================
class PortfolioEnv:
    def __init__(
        self,
        rets: pd.DataFrame,
        rets_norm: pd.DataFrame,
        window=12,
        trans_cost_bps=10.0,
        include_cash=True,
        seed=42,
        w_base_assets=None,
        base_cash=0.0,
        use_tilt_bounds=True,
        size_thresh=0.10,
        tilt_min_small=0.5,
        tilt_max_small=2.0,
        tilt_min_large=0.5,
        tilt_max_large=1.5,
        tilt_min_cash=0.5,
        tilt_max_cash=1.5,
        hysteresis_bps=10,
        risk_av=0.0,
        cov_win=12,
        risk_mode="diag",
        rf_per_period: float = 0.0,
    ):
        assert rets.shape == rets_norm.shape

        if rets.shape[0] <= 2:
            raise ValueError("Serie troppo corta per costruire l'ambiente (n<=2)")
        self.window = int(min(window, max(1, rets.shape[0] - 1)))

        self.rets_df = rets

        self.rets = np.nan_to_num(
            rets.values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0
        )
        self.rets_norm = np.nan_to_num(
            rets_norm.values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0
        )
        self.asset_names = list(rets.columns)
        self.n_assets = rets.shape[1]
        self.include_cash = include_cash
        self.cost = float(trans_cost_bps) / 1e4  # bps → frazione
        self.rng = np.random.RandomState(seed)
        self.use_tilt_bounds = use_tilt_bounds
        self.hyst_threshold = float(hysteresis_bps) / 1e4
        self.risk_av = float(risk_av)
        self.cov_win = int(max(2, cov_win))
        self.risk_mode = risk_mode.lower().strip()
        self.alpha = 2.0 / (self.cov_win + 1.0)
        self.rf_per_period = float(rf_per_period)

        if w_base_assets is None:
            w_base_assets = np.ones(self.n_assets) / self.n_assets
        self.w_base_assets = np.array(w_base_assets, dtype=float)
        self.base_cash = float(base_cash) if include_cash else 0.0

        if self.use_tilt_bounds:
            self.lb, self.ub = build_tilt_bounds(
                self.w_base_assets,
                self.base_cash,
                include_cash,
                size_thresh,
                tilt_min_small,
                tilt_max_small,
                tilt_min_large,
                tilt_max_large,
                tilt_min_cash,
                tilt_max_cash,
            )
        else:
            self.lb = np.zeros(self.n_assets + (1 if include_cash else 0))
            self.ub = np.ones_like(self.lb)

        self.t: Optional[int] = None
        self.prev_w: Optional[np.ndarray] = None
        self._reset_state()

    @property
    def action_dim(self):
        return self.n_assets + (1 if self.include_cash else 0)

    def _reset_state(self):
        self.t = self.window
        base = (
            np.concatenate([self.w_base_assets, [self.base_cash]])
            if self.include_cash else self.w_base_assets
        ).astype(np.float32)
        s = base.sum()
        if s <= 0:
            base = np.zeros(self.action_dim, np.float32)
            if self.include_cash:
                base[-1] = 1.0
            else:
                base[:] = 1.0 / self.action_dim
        else:
            base = (base / s).astype(np.float32)
        self.prev_w = base

    def reset(self) -> Tuple[np.ndarray, np.ndarray]:
        self._reset_state()
        return self._get_obs()

    def _get_obs(self):
        win = self.rets_norm[self.t - self.window : self.t, :]
        return win.copy(), self.prev_w[: self.n_assets].copy()

    def _ewma_cov(self, X: np.ndarray):
        X = np.nan_to_num(X.astype(float), nan=0.0, posinf=0.0, neginf=0.0)
        m, n = X.shape
        mu = np.zeros(n)
        C = np.zeros((n, n))
        w = 0.0
        a = self.alpha
        for i in range(m):
            x = X[i]
            mu = a * x + (1 - a) * mu
            d = (x - mu).reshape(-1, 1)
            C = a * (d @ d.T) + (1 - a) * C
            w = (1 - a) * w + a
        if w > 1e-12:
            C = C / w
        if self.risk_mode == "diag":
            C = np.diag(np.diag(C))
        return np.nan_to_num(C, nan=0.0, posinf=0.0, neginf=0.0)

    def _risk_penalty(self, w_action: np.ndarray) -> float:
        if self.risk_av <= 0:
            return 0.0
        t0 = max(0, self.t - self.cov_win)
        X = self.rets[t0 : self.t, :]
        if X.shape[0] < 2:
            return 0.0
        C = self._ewma_cov(X)
        w_assets = w_action[: self.n_assets].astype(float)
        risk = float(w_assets @ C @ w_assets)  # var per periodo
        if not np.isfinite(risk):
            risk = 0.0
        return self.risk_av * risk

    def _constrain_action(self, a_simplex: np.ndarray) -> np.ndarray:
        w_prop = (
            project_bounded_simplex(a_simplex, self.lb, self.ub, s=1.0)
            if self.use_tilt_bounds else a_simplex.copy()
        )
        turnover = float(np.abs(w_prop - self.prev_w).sum())
        if turnover < self.hyst_threshold:
            return self.prev_w.copy()
        return w_prop

    def step(self, a_simplex: np.ndarray):
        assert a_simplex.shape == (self.action_dim,)
        w_action = self._constrain_action(a_simplex)
        r_vec = self.rets[self.t, :]
        gross_ret = float(np.dot(w_action[: self.n_assets], r_vec))
        if self.include_cash:
            gross_ret += float(w_action[-1] * self.rf_per_period)
        tr_cost = self.cost * float(np.abs(w_action - self.prev_w).sum())
        risk_pen = self._risk_penalty(w_action)
        net_ret = gross_ret - tr_cost - risk_pen
        self.prev_w = w_action.astype(np.float32)
        self.t += 1
        done = self.t >= self.rets.shape[0]
        obs = self._get_obs() if not done else None
        info: Dict[str, float] = {
            "gross_ret": gross_ret,
            "cost": tr_cost,
            "risk_pen": risk_pen,
        }
        return obs, net_ret, done, info

# =========================
# 6) Actor-Critic
# =========================
import torch
import torch.nn as nn
from torch.distributions import Dirichlet

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
try:
    torch.use_deterministic_algorithms(False)
    torch.backends.cudnn.benchmark = True
except Exception:
    pass

class ActorCritic(nn.Module):
    def __init__(self, n_assets, action_dim, window=12, hidden=128):
        super().__init__()
        self.window = window
        self.n_assets = n_assets
        self.backbone = nn.Sequential(
            nn.LazyLinear(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
        )
        self.act_head = nn.Sequential(nn.Linear(hidden, action_dim), nn.Softplus())
        self.val_head = nn.Linear(hidden, 1)

    def forward(self, obs):
        rets_win, w_prev = obs
        B = rets_win.shape[0]
        x = torch.cat([rets_win.reshape(B, -1), w_prev], dim=1)
        x = torch.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
        h = self.backbone(x)
        alpha = self.act_head(h) + 1e-3        # alpha > 0
        alpha = torch.clamp(alpha, 1e-3, 1e6)  # stabilità
        value = self.val_head(h).squeeze(-1)
        return alpha, value

    def act(self, obs, deterministic=False):
        alpha, value = self.forward(obs)
        dist = Dirichlet(alpha)
        if deterministic:
            action = alpha / alpha.sum(dim=-1, keepdim=True)
            logp = dist.log_prob(action)
        else:
            action = dist.rsample()
            logp = dist.log_prob(action)
        entropy = dist.entropy()
        return action, logp, entropy, value

# =========================
# 7) Trainer (stateful rollout + bootstrap/GAE)
# =========================
@dataclass
class A2CConfig:
    window: int = WINDOW
    gamma: float = GAMMA
    lr: float = LR
    value_coef: float = VALUE_COEF
    entropy_coef: float = ENTROPY_COEF
    max_grad_norm: float = MAX_GRAD_NORM
    n_steps: int = N_STEPS
    total_updates: int = TOTAL_UPDATES
    trans_cost_bps: float = TRANS_COST_BPS
    include_cash: bool = INCLUDE_CASH
    seed: int = SEED
    use_tilt_bounds: bool = USE_TILT_BOUNDS
    size_thresh: float = SIZE_THRESH
    tilt_min_small: float = TILT_MIN_SMALL
    tilt_max_small: float = TILT_MAX_SMALL
    tilt_min_large: float = TILT_MIN_LARGE
    tilt_max_large: float = TILT_MAX_LARGE
    tilt_min_cash: float = TILT_MIN_CASH
    tilt_max_cash: float = TILT_MAX_CASH
    hysteresis_bps: int = HYSTERESIS_BPS
    w_base_assets: np.ndarray = None
    base_cash: float = 0.0
    risk_av: float = RISK_AV
    cov_win: int = COV_WIN
    risk_mode: str = RISK_MODE
    rf_per_period: float = RF_PER_PERIOD
    use_gae: bool = USE_GAE
    gae_lambda: float = GAE_LAMBDA

class Trajectory(NamedTuple):
    obs_rets: torch.Tensor
    obs_wprev: torch.Tensor
    actions: torch.Tensor
    logps: torch.Tensor
    rewards: torch.Tensor
    dones: torch.Tensor
    values: torch.Tensor
    entropies: torch.Tensor
    last_obs_rets: torch.Tensor
    last_obs_wprev: torch.Tensor
    last_done: torch.Tensor

def to_tensor_obs(batch_obs):
    rets_win, w_prev = zip(*batch_obs)
    rw = torch.tensor(np.stack(rets_win), dtype=torch.float32, device=DEVICE)
    wp = torch.tensor(np.stack(w_prev), dtype=torch.float32, device=DEVICE)
    return (
        torch.nan_to_num(rw, nan=0.0, posinf=0.0, neginf=0.0),
        torch.nan_to_num(wp, nan=0.0, posinf=0.0, neginf=0.0),
    )

def make_env(rets, retsn, cfg: A2CConfig):
    return PortfolioEnv(
        rets,
        retsn,
        window=cfg.window,
        trans_cost_bps=cfg.trans_cost_bps,
        include_cash=cfg.include_cash,
        seed=cfg.seed,
        w_base_assets=(cfg.w_base_assets if cfg.w_base_assets is not None else w_base_assets),
        base_cash=cfg.base_cash,
        use_tilt_bounds=cfg.use_tilt_bounds,
        size_thresh=cfg.size_thresh,
        tilt_min_small=cfg.tilt_min_small,
        tilt_max_small=cfg.tilt_max_small,
        tilt_min_large=cfg.tilt_min_large,
        tilt_max_large=cfg.tilt_max_large,
        tilt_min_cash=cfg.tilt_min_cash,
        tilt_max_cash=cfg.tilt_max_cash,
        hysteresis_bps=cfg.hysteresis_bps,
        risk_av=cfg.risk_av,
        cov_win=cfg.cov_win,
        risk_mode=cfg.risk_mode,
        rf_per_period=cfg.rf_per_period,
    )

def rollout(env: PortfolioEnv, model: ActorCritic, n_steps: int) -> Trajectory:
    if env.t is None or env.t >= env.rets.shape[0]:
        obs = env.reset()
    else:
        obs = env._get_obs()

    traj = []
    for _ in range(n_steps):
        rw, wp = to_tensor_obs([obs])
        a, lp, ent, val = model.act((rw, wp), deterministic=False)
        nxt, r, d, _ = env.step(a.squeeze(0).detach().cpu().numpy())
        traj.append((obs, a.squeeze(0), lp.squeeze(0), float(r), d, val.squeeze(0), ent.squeeze(0)))
        obs = env.reset() if d else nxt

    obs_rets = torch.tensor(np.stack([t[0][0] for t in traj]), dtype=torch.float32, device=DEVICE)
    obs_wprev = torch.tensor(np.stack([t[0][1] for t in traj]), dtype=torch.float32, device=DEVICE)
    actions = torch.stack([t[1] for t in traj]).to(DEVICE)
    logps = torch.stack([t[2] for t in traj]).to(DEVICE)
    rewards = torch.tensor([t[3] for t in traj], dtype=torch.float32, device=DEVICE)
    dones = torch.tensor([t[4] for t in traj], dtype=torch.float32, device=DEVICE)
    values = torch.stack([t[5] for t in traj]).to(DEVICE)
    entropies = torch.stack([t[6] for t in traj]).to(DEVICE)

    if env.t is None or env.t >= env.rets.shape[0]:
        last_obs = env.reset()
        last_done = 1.0
    else:
        last_obs = env._get_obs()
        last_done = 0.0

    last_rw, last_wp = to_tensor_obs([last_obs])

    return Trajectory(
        obs_rets, obs_wprev, actions, logps, rewards, dones, values, entropies,
        last_rw.squeeze(0), last_wp.squeeze(0),
        torch.tensor(last_done, dtype=torch.float32, device=DEVICE),
    )

def compute_bootstrapped_returns(rewards, dones, last_value, gamma):
    G = torch.zeros_like(rewards)
    run = last_value
    for t in reversed(range(len(rewards))):
        run = rewards[t] + gamma * run * (1.0 - dones[t])
        G[t] = run
    return G

def compute_gae(rewards, dones, values, last_value, gamma, lam):
    T = len(rewards)
    adv = torch.zeros_like(rewards)
    next_value = last_value
    gae = 0.0
    for t in reversed(range(T)):
        delta = rewards[t] + gamma * next_value * (1.0 - dones[t]) - values[t]
        gae = delta + gamma * lam * (1.0 - dones[t]) * gae
        adv[t] = gae
        next_value = values[t]
    returns = adv + values
    return adv, returns

def train_a2c(rets: pd.DataFrame, retsn: pd.DataFrame, cfg: A2CConfig):
    env = make_env(rets, retsn, cfg)
    model = ActorCritic(n_assets=rets.shape[1], action_dim=env.action_dim, window=cfg.window).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr)

    for upd in range(1, cfg.total_updates + 1):
        traj = rollout(env, model, cfg.n_steps)
        with torch.no_grad():
            _, last_v = model.forward((traj.last_obs_rets.unsqueeze(0), traj.last_obs_wprev.unsqueeze(0)))
            last_v = last_v.squeeze(0)

        if cfg.use_gae:
            adv, returns = compute_gae(traj.rewards, traj.dones, traj.values, last_v, cfg.gamma, cfg.gae_lambda)
        else:
            returns = compute_bootstrapped_returns(traj.rewards, traj.dones, last_v, cfg.gamma)
            adv = returns - traj.values

        actor_loss = -(traj.logps * adv.detach()).mean() - cfg.entropy_coef * traj.entropies.mean()
        value_loss = cfg.value_coef * (traj.values - returns.detach()).pow(2).mean()
        loss = actor_loss + value_loss

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
        opt.step()

        if upd % 50 == 0 or upd == 1:
            avg_r = float(traj.rewards.mean().item())
            print(f"[upd {upd}] loss={loss.item():.4f}  actor={actor_loss.item():.4f}  value={value_loss.item():.4f}  avg_r={avg_r:.6f}")
    return model

# =========================
# 8) Metriche & Valutazione
# =========================
import matplotlib.pyplot as plt

def compute_equity_curve(rs: pd.Series, start_at: float = 1.0):
    return (1 + rs).cumprod() * start_at

def annualized_sharpe(rs: pd.Series, ppy: int = PERIODS_PER_YEAR, rf: float = 0.0):
    ex = rs - (rf / ppy)
    mu = ex.mean() * ppy
    sd = ex.std(ddof=1) * np.sqrt(ppy)
    return np.nan if sd == 0 else float(mu / sd)

def max_drawdown(eq: pd.Series):
    dd = eq / eq.cummax() - 1.0
    return float(dd.min()), dd

def annualized_cagr(eq: pd.Series, ppy: int = PERIODS_PER_YEAR) -> float:
    """CAGR annualizzato dall'equity curve.
    Se l'indice è DatetimeIndex usa gli anni effettivi, altrimenti scala con PERIODS_PER_YEAR."""
    if len(eq) < 2:
        return np.nan
    if isinstance(eq.index, pd.DatetimeIndex):
        years = (eq.index[-1] - eq.index[0]).days / 365.25
        if years <= 0:
            return np.nan
        return float((eq.iloc[-1] / eq.iloc[0]) ** (1.0 / years) - 1.0)
    else:
        n = len(eq)
        return float((eq.iloc[-1] / eq.iloc[0]) ** (ppy / n) - 1.0)

def evaluate_model(
    model: ActorCritic,
    rets: pd.DataFrame,
    retsn: pd.DataFrame,
    cfg: A2CConfig,
    title_suffix: str = "",
    base_weights_assets: Optional[np.ndarray] = None,
    base_cash: float = 0.0,
    over_under_threshold: float = 0.05,   #
    env = make_env(rets, retsn, cfg)
    obs = env.reset()
    rs = []
    ws = []
    gross_list, cost_list, risk_list = [], [], []
    while True:
        rw, wp = to_tensor_obs([obs])
        with torch.no_grad():
            a, _, _, _ = model.act((rw, wp), deterministic=True)
        w = a.squeeze(0).cpu().numpy()
        ws.append(w)
        obs, r, d, info = env.step(w)
        rs.append(r)
        gross_list.append(info["gross_ret"])
        cost_list.append(info["cost"])
        risk_list.append(info["risk_pen"])
        if d:
            break
    idx = rets.index[-len(rs):]
    rs = pd.Series(rs, index=idx, name="A2C_returns")
    eq = compute_equity_curve(rs, 1.0)
    sh = annualized_sharpe(rs, PERIODS_PER_YEAR)
    mdd, dd = max_drawdown(eq)
    cagr = annualized_cagr(eq, PERIODS_PER_YEAR)
    avg_turnover = float(np.abs(np.diff(np.vstack(ws), axis=0)).sum(axis=1).mean()) if len(ws) > 1 else 0.0

    # Grafico equity
    plt.figure()
    eq.plot(title=f"Equity Curve (A2C) {title_suffix}".strip())
    plt.xlabel("Data"); plt.ylabel("Equity")
    plt.show()

    # Grafico drawdown
    plt.figure()
    dd.plot(title=f"Drawdown (A2C) {title_suffix}".strip())
    plt.xlabel("Data"); plt.ylabel("Drawdown")
    plt.show()

    # Grafico pesi (stacked area)
    w_cols = env.asset_names + (["CASH"] if cfg.include_cash else [])
    w_df = pd.DataFrame(ws, index=idx, columns=w_cols)
    plt.figure()
    w_df.plot.area(title=f"Pesi Allocati (A2C) {title_suffix}".strip(), alpha=0.8)
    plt.xlabel("Data"); plt.ylabel("Peso")
    plt.legend(loc="upper right", ncol=2)
    plt.show()

    print(f"Punti: {len(eq)}  | CAGR: {cagr:.2%}  | Sharpe: {sh:.3f}  | MaxDD: {mdd:.2%}  | Turnover medio: {avg_turnover:.3f}  | {title_suffix}")
    print(f"Gross ret medio/periodo: {np.mean(gross_list):.6f}  | Costi medi: {np.mean(cost_list):.6f}  | Penalità rischio media: {np.mean(risk_list):.6f}")

# =========================
# 9) Train & Grafici
# =========================
cfg = A2CConfig(w_base_assets=w_base_assets, base_cash=base_cash)

# Train su TRAIN
model = train_a2c(rets_train, retsn_train, cfg)

# Val + Test (out-of-train)
eval_rets_vt = pd.concat([rets_val, rets_test])
eval_retsn_vt = pd.concat([retsn_val, retsn_test])
_, _, _, tilt_table_vt, metrics_vt = evaluate_model(
    model, eval_rets_vt, eval_retsn_vt, cfg, title_suffix="— Val + Test (robust)",
    base_weights_assets=w_base_assets, base_cash=base_cash, over_under_threshold=0.05
)

# Tutto (diagnostico)
_, _, _, tilt_table_all, metrics_all = evaluate_model(
    model, rets, rets_norm_all, cfg, title_suffix="— Tutto (diagnostico)",
    base_weights_assets=w_base_assets, base_cash=base_cash, over_under_threshold=0.05
)

# =========================
# 10) Save
# =========================
import torch as _torch
_torch.save(model.state_dict(), MODEL_PATH)
print("Modello salvato in:", MODEL_PATH)
